# Bottleneck units

Did grid-like units emerge in the linear layer? The paper's Fig. 1d,g — 129 of
512 units with gridness > 0.37.

Look at a handful of units first, then export the full PDF only once they look
right. No untrained control and no position decoding here; that is
`03_path_integration.ipynb`. Border cells, conjunctive cells, grid scale and
stability are `05_cell_types.ipynb`. Nothing is implemented in this notebook —
scoring comes from `scores.py`, plots from `figures.py`.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import numpy as np
import torch
import matplotlib.pyplot as plt

import evaluate
import figures
from config import Config
from dataset import build_dataloader
from scores import score_units

CHECKPOINT = "data/checkpoints/baseline/checkpoint_epoch299.pt"
N_TRAJECTORIES = 4000
NBINS = 32  # paper's Methods: 32x32 spatial bins

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = Config()
print(device, "|", ROOT)

In [ ]:
# Inference once (~1 min for 4000 trajectories); every cell below reuses it.
model, _, epoch, place, hd = evaluate.load_models(cfg, CHECKPOINT, device)

loader = build_dataloader("data/shards", shard_indices=None,
                          batch_size=cfg.train.minibatch_size, shuffle=True)
batches, n_traj = evaluate.collect_batches(loader, N_TRAJECTORIES)
true_pos = np.concatenate([b["target_pos"].numpy() for b in batches], axis=0)
headings = np.concatenate([b["target_hd"].numpy() for b in batches]).reshape(-1)

_, bottleneck_acts, lstm_acts = evaluate.run_model(
    model, batches, place, hd, device, want_activations=True)

print(f"epoch {epoch} | {n_traj} trajectories x {true_pos.shape[1]} steps")
print(f"bottleneck {bottleneck_acts.shape} | lstm {lstm_acts.shape}")

## Score every unit

One ratemap and one spatial autocorrelogram per unit; gridness is the best score
over the paper's expanding annulus (outer radius 8→20 bins, step 2). 640 units
in total, so this is the slow cell — a couple of minutes.

In [ ]:
scorer = evaluate.build_scorer(cfg, NBINS)
xy = true_pos.reshape(-1, 2)

scored = {}  # layer -> (scores_60, ratemaps, sacs, mask_params)
tuned = {}   # layer -> (hd_tuning [n_units, 20], resultant lengths)
for name, acts in [("bottleneck", bottleneck_acts), ("lstm", lstm_acts)]:
    flat = acts.reshape(-1, acts.shape[-1])
    print(f"scoring {flat.shape[1]} {name} units ...")
    scored[name] = score_units(scorer, xy, flat)
    evaluate.report_gridness(name, scored[name][0])
    tuned[name] = evaluate.score_directional(headings, flat)
    evaluate.report_directional(name, tuned[name][1])
print("\npaper: gridness 129/512 (25.2%) in the bottleneck, and grid-like units\n"
      "are specific to it -- the raw recurrent state should score ~0.\n"
      "paper: head-direction 10.2% of linear-layer units above 0.47.")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
figures.plot_score_distribution({k: v[0] for k, v in scored.items()},
                                evaluate.GRIDNESS_THRESHOLD, ax=ax)
ax.set_title(f"gridness distribution, epoch {epoch}")
plt.show()

## Look at a few units

The full Fig. 1d layout: ratemap on top, autocorrelogram below it with the
winning annulus drawn on, activity vs head direction as a polar plot at the
bottom. Titles are `unit (gridness)`; red = above 0.37.

The polar row is shifted by each unit's own minimum before drawing — a linear
layer's activations are signed, and a negative radius is not something a polar
axis can show. Same convention as the resultant-vector measure.

In [ ]:
LAYER = "bottleneck"
TOP_N = 8

scores_60, ratemaps, sacs, mask_params = scored[LAYER]
hd_tuning = tuned[LAYER][0]
top = np.argsort(-scores_60)[:TOP_N]
print("unit: gridness  ", {int(i): round(float(scores_60[i]), 2) for i in top})

fig = figures.plot_unit_grid(
    ratemaps, sacs, mask_params, scores_60, NBINS,
    indices=top, cols=TOP_N, threshold=evaluate.GRIDNESS_THRESHOLD,
    hd_tuning=hd_tuning, panel_size=2.0,
    title=f"{LAYER}: top {TOP_N} by gridness (epoch {epoch})  "
          f"[ratemap / autocorrelogram / head direction]")
plt.show()

In [ ]:
# Any units, in any order. Default: the ones straddling the cutoff, to see what
# 0.37 is actually separating.
order = np.argsort(-scores_60)
n_pass = int((scores_60 > evaluate.GRIDNESS_THRESHOLD).sum())
UNITS = [int(i) for i in order[max(0, n_pass - 4):n_pass + 4]]

fig = figures.plot_unit_grid(
    ratemaps, sacs, mask_params, scores_60, NBINS,
    indices=UNITS, cols=len(UNITS), threshold=evaluate.GRIDNESS_THRESHOLD,
    hd_tuning=hd_tuning, panel_size=2.0,
    title=f"{LAYER}: units straddling the 0.37 cutoff")
plt.show()

## Export every unit

Only worth running once the previews look right — 512 units is a large PDF and
takes a while to render.

In [ ]:
OUT_DIR = Path(evaluate.default_out_dir(CHECKPOINT))
OUT_DIR.mkdir(parents=True, exist_ok=True)

for name, (s, rm, sc, mp) in scored.items():
    path = figures.save_unit_pdf(
        str(OUT_DIR / f"{name}_ratemaps_epoch{epoch}.pdf"),
        ratemaps=rm, sacs=sc, mask_params=mp, scores=s, nbins=NBINS, cols=16,
        threshold=evaluate.GRIDNESS_THRESHOLD, hd_tuning=tuned[name][0],
        title=f"{name} units, epoch {epoch} "
              f"(red = gridness > {evaluate.GRIDNESS_THRESHOLD})")
    print("saved", path)